# Pipeline

## Run (Default Catalog)

### Run - Catalog Processing

In [ ]:
import catalog_processing

catalog_processing.main()

### Run - Pipeline


In the next cell, provide the OpenRouter API key required to call the language models used in the pipeline

In [ ]:
OPENROUTER_API_KEY = None

In [ ]:
import json
import os
import phase_1
import phase_2
import phase_3

from phase_utils import model_info

models = {
    'model_1': {'model_name': 'deepseek-v3.2', 'model_id': 'deepseek/deepseek-v3.2', 'model_provider': 'atlas-cloud/fp8', 'model_reasoning': 'Token'},
    'model_2': {'model_name': 'gemini-2.5-flash', 'model_id': 'google/gemini-2.5-flash', 'model_provider': 'google-vertex/global', 'model_reasoning': 'Token'},
    'model_3': {'model_name': 'gpt-oss-120b', 'model_id': 'openai/gpt-oss-120b', 'model_provider': 'siliconflow/fp8', 'model_reasoning': 'Effort'}
}

data_source = 'Catalogo Egizio ITA - Max 5000 Tokens'
percentage = 100

# Pipeline:
for model in models.values():

    model_info(model)
    phase_1.main(data_source = data_source, model = model, percentage=percentage, openrouter_api_key = OPENROUTER_API_KEY, default_choices = True)
    phase_2.main(model = model, openrouter_api_key = OPENROUTER_API_KEY, default_choices = True)
    phase_3.main(model = model, openrouter_api_key = OPENROUTER_API_KEY, default_choices = True)

# Final Result:
all_stories = {}

for model in models.values():
    results_folder = os.path.join('results', model['model_name'], 'phase_3', 'processed_results')

    for filename in os.listdir(results_folder):
        with open(os.path.join(results_folder, filename), 'r', encoding='utf-8') as file:
            all_stories.update(json.load(file))

with open('results/all_stories.json', 'w', encoding='utf-8') as out_file:
    json.dump(all_stories, out_file, ensure_ascii=False, indent=4)

## Run (Custom Catalog)

### Configuration

In the next cell, specify the configuration required to run `catalog_processing.py` with a custom catalog:

<br>
<b>1. Input catalog and output settings</b>
<ul style="line-height:1.8;">
  <li><b><span style="color:#2E86C1;">catalog_name</span></b> - Name of the Excel file stored in <code>resources/catalog/original/</code></li>
  <li><b><span style="color:#2E86C1;">sheet_name</span></b> - Name of the Excel sheet to process</li>
  <li><b><span style="color:#2E86C1;">filtered_name</span></b> - Name of the filtered catalog file that will be saved</li>
  <li><b><span style="color:#2E86C1;">tokens_per_file</span></b> - Maximum number of tokens allowed in each processed catalog chunk</li>
  <li><b><span style="color:#2E86C1;">processed_name</span></b> - Name of the output folder containing the processed catalog chunks</li>
</ul>
<br>
<b>2. Column mapping to the internal schema</b>
<ul style="line-height:1.8;">
  <li><b><span style="color:#2E86C1;">id_column</span></b> — Name of the column containing the unique identifier of each catalog record</li>
  <li><b><span style="color:#2E86C1;">description_column</span></b> — Name of the column containing the main textual description</li>
  <li><b><span style="color:#2E86C1;">curatorial_description_column</span></b> — Name of the column containing the curatorial description. Leave it empty if the catalog does not include one</li>
</ul>

In [ ]:
catalog_name = ''
sheet_name = ''
filtered_name = ''
tokens_per_file = 5000
processed_name = ''

id_column = ''
description_column = ''
curatorial_description_column = ''

The filtering function should remove records that should not be processed and then convert the catalog to the internal schema expected by the pipeline:

<ul style="line-height:1.8;">
  <li><b><span style="color:#2E86C1;">Catalog_ID</span></b> - Unique identifier of the catalog record</li>
  <li><b><span style="color:#2E86C1;">Description</span></b> - Main description</li>
  <li><b><span style="color:#2E86C1;">Curator_Description</span></b> - Optional curatorial description</li>
</ul>

This function is intended to be customized for each dataset. Therefore, the filtering criteria should be defined according to the structure and content of the input catalog.

In the next cell, define a filtering function adapted to the structure of the input dataset.

The filtering operations provided in `catalog_processing.py` were designed for the catalog used in the experimental evaluation. However, they can be adapted, replaced, or extended, and may serve as a useful reference for identifying which records should be excluded.

For each filtering operation, store the number of removed records in a dedicated variable. These variables should then be included in the final print statement, so that the filtering process can be reported clearly and reproducibly.

Example:

```python
rows_before = len(dataframe)
dataframe = <filtering operation applied to the dataframe>
rows_removed_<filter_category> = rows_before - len(dataframe)
```

In [ ]:
import os
import os.path as path
import pandas

# Print:
from colorama import Fore, Style
Y = Fore.YELLOW
C = Fore.CYAN
R = Style.RESET_ALL

def custom_catalog_filtering(dataframe, filtered_folder, filtered_name):

    os.makedirs(filtered_folder, exist_ok=True) 
    rows_original = len(dataframe)

    # Filtering Operation 01:
    rows_before = len(dataframe) 
    # dataframe = '<filtering operation applied to the dataframe>'
    rows_removed_filter_category_01 = rows_before - len(dataframe)

    # Filtering Operation 02:
    rows_before = len(dataframe) 
    # dataframe = '<filtering operation applied to the dataframe>'
    rows_removed_filter_category_02 = rows_before - len(dataframe)

    # ...


    # Statistics:
    print(f"""
          
{Y}CATALOG FILTERING{R}
          
Rows in the original dataset: {C}{rows_original}{R}
Rows in the filtered dataset: {C}{len(dataframe)}{R}

Removals:
• {C}{rows_removed_filter_category_01}{R} rows removed due to...
• {C}{rows_removed_filter_category_02}{R} rows removed due to...
    """)


    # Rename columns to match the internal schema expected by the pipeline:
    dataframe = dataframe.rename(columns={
        id_column: 'Catalog_ID',
        description_column: 'Description',
        curatorial_description_column: 'Curator_Description'
        })

    if 'Curator_Description' not in dataframe.columns:
        dataframe['Curator_Description'] = ''

    # Keep only the columns required by the pipeline:
    dataframe = dataframe[['Catalog_ID', 'Description', 'Curator_Description']]

    # Adjustments for JSON conversion:
    dataframe = dataframe.where(pandas.notna(dataframe), None)
    dataframe.columns = dataframe.columns.str.lower()

    file_path = path.join(filtered_folder, filtered_name)

    if path.exists(file_path):
        os.remove(file_path)

    dataframe.to_excel(file_path, index=False)

    return dataframe

### Run - Catalog Processing

The following cells execute the catalog processing step by step using the custom filtering function defined above.
This reproduces the logic of `catalog_processing.main()`, but replaces the default filtering function with `custom_catalog_filtering()`.

In [ ]:
import catalog_processing
import os.path as path
import pandas

original_catalog = path.join('resources', 'catalog', 'original', catalog_name)
filtered_folder = path.join('resources', 'catalog', 'filtered')
processed_folder = path.join('resources', 'catalog', 'processed')

# Dataframe:
dataframe = pandas.read_excel(original_catalog, sheet_name=sheet_name)
dataframe_filtered = custom_catalog_filtering(dataframe, filtered_folder, filtered_name)

# Indexing and Conversion:
catalog_processing.catalog_indexing(dataframe_filtered, filtered_folder)
catalog_processing.catalog_conversion(dataframe_filtered, path.join(processed_folder,processed_name), tokens_per_file)

### Run - Pipeline 

In the next cell, define the configuration required to run the pipeline on the selected processed catalog and models.
<br>
<ul style="line-height:1.8;">
  <li><b><span style="color:#2E86C1;">OPENROUTER_API_KEY</span></b> — OpenRouter API key used to call the selected language models. Replace the placeholder value with your own API key before running the pipeline.</li>

  <li><b><span style="color:#2E86C1;">data_source</span></b> — Name of the processed catalog folder generated by Catalog Processing. This value must match the folder stored in <code>resources/catalog/processed/</code>.</li>

  <li><b><span style="color:#2E86C1;">percentage</span></b> — Percentage of catalog files to process. Use <code>100</code> to process the full catalog, or a smaller value for testing.</li>

  <li><b><span style="color:#2E86C1;">models</span></b> — Dictionary containing the models to be used in the experimentation. Each model entry includes the model name, OpenRouter model ID, provider, and reasoning configuration.</li>
</ul>

<br>
Each model should be defined using the following fields:

<ul style="line-height:1.8;">
  <li><b><span style="color:#2E86C1;">model_name</span></b> — Short name used to identify the model in the output folders.</li>
  <li><b><span style="color:#2E86C1;">model_id</span></b> — OpenRouter model identifier.</li>
  <li><b><span style="color:#2E86C1;">model_provider</span></b> — Provider selected for the model execution.</li>
  <li><b><span style="color:#2E86C1;">model_reasoning</span></b> — Reasoning configuration used by the API call. Use <code>Token</code> for token-based reasoning or <code>Effort</code> for effort-based reasoning.</li>
</ul>

In [ ]:
# OperRouter:
OPENROUTER_API_KEY = None

# Sperimentation setup:
data_source = ''
percentage = 100

# Model:
models = {
    'model_1': {'model_name': 'deepseek-v3.2', 'model_id': 'deepseek/deepseek-v3.2', 'model_provider': 'atlas-cloud/fp8', 'model_reasoning': 'Token'},
    'model_2': {'model_name': 'gemini-2.5-flash', 'model_id': 'google/gemini-2.5-flash', 'model_provider': 'google-vertex/global', 'model_reasoning': 'Token'},
    'model_3': {'model_name': 'gpt-oss-120b', 'model_id': 'openai/gpt-oss-120b', 'model_provider': 'siliconflow/fp8', 'model_reasoning': 'Effort'}
}

In [ ]:
import json
import os
import phase_1
import phase_2
import phase_3

from phase_utils import model_info

# Pipeline:
for model in models.values():

    model_info(model)
    phase_1.main(data_source = data_source, model = model, percentage=percentage, openrouter_api_key = OPENROUTER_API_KEY, default_choices = True)
    phase_2.main(model = model, openrouter_api_key = OPENROUTER_API_KEY, default_choices = True)
    phase_3.main(model = model, openrouter_api_key = OPENROUTER_API_KEY, default_choices = True)

# Final Result:
all_stories = {}

for model in models.values():
    results_folder = os.path.join('results', model['model_name'], 'phase_3', 'processed_results')

    for filename in os.listdir(results_folder):
        with open(os.path.join(results_folder, filename), 'r', encoding='utf-8') as file:
            all_stories.update(json.load(file))

with open('results/all_stories.json', 'w', encoding='utf-8') as out_file:
    json.dump(all_stories, out_file, ensure_ascii=False, indent=4)